# Notebook 08 — Training Final: Best Hyperparameter Random Search (FinBERT-LSTM, OHLCV + Sentimen)

**Tujuan notebook ini:** tempat pengujian model dengan melatih ULANG (1x, penuh) memakai konfigurasi
hyperparameter TERBAIK yang sudah ditemukan Random Search 20-iterasi di
`07_LSTM_FinBERT_skripsi_scoped.ipynb` — bukan mengulang proses pencarian (search), tapi menjalankan
training final yang lebih mendalam untuk kebutuhan pelaporan BAB IV: learning curve per-epoch (train
vs val), evaluasi test-set, dan artefak model yang bisa dipakai ulang.

**Model & data:** FinBERT-LSTM (**Model B**) — fitur **OHLCV + `sentiment_mean`** (6 kolom,
`FEATURE_COLS` dari `pipeline/feature_engineering/dataset_builder.py`). Horizon prediksi **1 hari**
(`label = 1 jika Close[t+1] > Close[t]`), sesuai Batasan Masalah BAB I skripsi — sama persis dengan
scope notebook 07.

**Sumber best config:** dibaca LANGSUNG dari artefak hasil notebook 07
(`data/artifacts/lstm_weights/finbert_lstm_skripsi_metadata.json`) — bukan hardcode manual, supaya
notebook ini selalu sinkron dengan hasil Random Search terbaru kalau notebook 07 dijalankan ulang.

**Reuse (TIDAK reinvent), konsisten prinsip project ini:**
- `pipeline/feature_engineering/dataset_builder.py` → `load_joined()`, `FEATURE_COLS`.
- `models/lstm/lstm_model.py` → `FlexibleLSTMClassifier` (arsitektur PERSIS sama dengan notebook 07 —
  2-class softmax, aktivasi FC-head configurable, last-timestep output).
- `models/lstm/notebook07_common.py` → `load_split_capped()` (split kronologis 70/15/15 + capping
  outlier IQR fit-train-only), `compute_class_weights()`, `build_optimizer()` — SATU-SATUNYA salinan
  logic ini di project (dipakai juga oleh API training live).

**Evaluasi:** HANYA Accuracy, Precision, Recall, F1-Score (`sklearn.metrics`), konsisten Batasan
Masalah BAB I — TIDAK menambah metrik lain (AUC-ROC dst di luar scope).

**Anti p-hacking:** hasil dilaporkan apa adanya. Metadata notebook 07 mencatat `recall=1.0` pada
model terbaik (indikasi model condong/collapse memprediksi kelas "naik"). Notebook ini TIDAK mencoba
mengakali angka tersebut supaya terlihat lebih baik, hanya melatih ulang & melaporkan hasil aktual.

**Status: notebook ini ditulis LENGKAP tapi BELUM DIEKSEKUSI.** Semua cell `outputs=[]`. Eksekusi
disarankan lewat `papermill`/`nbconvert --execute` headless (bukan cell-by-cell Jupyter MCP interaktif)
untuk hemat token, sebagai langkah terpisah setelah notebook ini direview.

## Bagian 1 — Setup & Load Best Config

Import reuse dari `pipeline/feature_engineering/dataset_builder.py`, `models/lstm/lstm_model.py`, dan
`models/lstm/notebook07_common.py`. Best hyperparameter config dibaca dari metadata JSON hasil Random
Search notebook 07 (`finbert_lstm_skripsi_metadata.json`).

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
if (PROJECT_ROOT / "pipeline").exists() is False:
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

import logging
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from pipeline.feature_engineering.dataset_builder import FEATURE_COLS, load_joined
from models.lstm.lstm_model import FlexibleLSTMClassifier
from models.lstm.notebook07_common import (
    build_optimizer,
    compute_class_weights,
    load_split_capped,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("notebook08")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Device: %s", DEVICE)


def set_seed(seed: int) -> None:
    """Set semua seed relevan untuk reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
RANDOM_SEARCH_METADATA_PATH = (
    PROJECT_ROOT / "data" / "artifacts" / "lstm_weights" / "finbert_lstm_skripsi_metadata.json"
)

with open(RANDOM_SEARCH_METADATA_PATH, "r", encoding="utf-8") as f:
    random_search_metadata = json.load(f)

BEST_CONFIG = random_search_metadata["config"]
WINDOW = random_search_metadata["window"]
HORIZON = random_search_metadata["horizon"]
TRAIN_SEED = 42  # sama dengan seed dasar Random Search notebook 07 (iterasi pemenang: seed 42 + i)

print("=== Best config hasil Random Search (notebook 07, Model B \u2014 FinBERT-LSTM) ===")
for k, v in BEST_CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nwindow = {WINDOW} | horizon = {HORIZON} hari")
print(f"best_val_f1 (saat search) = {random_search_metadata['best_val_f1']}")
print(f"best_val_acc (saat search) = {random_search_metadata['best_val_acc']}")
print(f"metrics test-set (saat search) = {random_search_metadata['metrics']}")

## Bagian 2 — Data: Load, Split 70/15/15, Capping Outlier

Reuse `load_split_capped()` dari `models/lstm/notebook07_common.py` — fungsi yang SAMA dipakai
notebook 07 & API training live (`training_service.py`). Melakukan: load `load_joined()` (OHLCV +
`sentiment_mean`) → label horizon-`HORIZON` hari → split kronologis persentase 70/15/15 → capping
outlier IQR (`Volume` & `sentiment_mean`, fit-train-only, anti-leakage).

In [ ]:
CAPPING_COLS = ["Volume", "sentiment_mean"]

df_train, df_val, df_test, capping_bounds, n_capped = load_split_capped(
    load_joined, CAPPING_COLS, horizon=HORIZON
)

print("=== Ukuran split (baris) ===")
print(f"  train: {len(df_train)} | val: {len(df_val)} | test: {len(df_test)}")

print("\n=== Label mean (proporsi 'naik') per split ===")
print(f"  train: {df_train['label'].mean():.4f}")
print(f"  val:   {df_val['label'].mean():.4f}")
print(f"  test:  {df_test['label'].mean():.4f}")

print("\n=== Batas capping IQR (dihitung dari train) ===")
for col, (lo, hi) in capping_bounds.items():
    print(f"  {col}: [{lo:.4f}, {hi:.4f}]")

print("\n=== Jumlah nilai yang di-cap per split ===")
print(pd.DataFrame(n_capped).T)

## Bagian 3 — Scaling & Windowing

`MinMaxScaler` fit HANYA di train (anti-leakage), lalu transform val/test. Sliding window
**fixed** sesuai `WINDOW` yang tercatat di metadata Random Search (window TIDAK ikut di-*search*,
konsisten Batasan Masalah BAB I) — windowing dilakukan per-split (tidak melintasi batas
train/val/test).

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
train_feat = scaler.fit_transform(df_train[FEATURE_COLS].values.astype(np.float64))
val_feat = scaler.transform(df_val[FEATURE_COLS].values.astype(np.float64))
test_feat = scaler.transform(df_test[FEATURE_COLS].values.astype(np.float64))

print("Scaled shapes \u2014 train:", train_feat.shape, "| val:", val_feat.shape, "| test:", test_feat.shape)

In [ ]:
def make_windows(features: np.ndarray, labels: np.ndarray, window: int):
    n = len(features)
    if n < window:
        return (
            np.empty((0, window, features.shape[1]), dtype=np.float32),
            np.empty((0,), dtype=np.int64),
        )
    num_samples = n - window + 1
    X = np.stack([features[i : i + window] for i in range(num_samples)]).astype(np.float32)
    y = labels[window - 1 : window - 1 + num_samples].astype(np.int64)
    return X, y


X_train, y_train = make_windows(train_feat, df_train["label"].values, WINDOW)
X_val, y_val = make_windows(val_feat, df_val["label"].values, WINDOW)
X_test, y_test = make_windows(test_feat, df_test["label"].values, WINDOW)

print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)

## Bagian 4 — Training Final (1x Run Penuh, Best Config)

Berbeda dari notebook 07 (tiap konfigurasi Random Search dilatih lalu model-nya DIBUANG kecuali yang
terbaik, history per-epoch tidak disimpan), di sini model terbaik dilatih ULANG **satu kali** dengan
history train/val (loss, accuracy, F1) dicatat **tiap epoch** — supaya learning curve bisa dianalisis
untuk BAB IV (indikasi overfitting/underfitting).

Class-weight (inverse-frequency, dihitung dari train saja) tetap dipakai, konsisten dengan
`run_notebook07_training()`.

In [ ]:
set_seed(TRAIN_SEED)

model = FlexibleLSTMClassifier(
    input_size=len(FEATURE_COLS),
    hidden_size=BEST_CONFIG["hidden_size"],
    num_layers=BEST_CONFIG["num_layers"],
    dropout=BEST_CONFIG["dropout"],
    activation=BEST_CONFIG["activation"],
    num_classes=2,
).to(DEVICE)

optimizer = build_optimizer(BEST_CONFIG["optimizer"], model.parameters(), BEST_CONFIG["lr"])
class_weights = compute_class_weights(y_train, DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(model)
print("\nClass weights (inverse-frequency, dari train):", class_weights.cpu().numpy())

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
train_loader = DataLoader(train_ds, batch_size=BEST_CONFIG["batch_size"], shuffle=True)

X_val_t = torch.from_numpy(X_val).to(DEVICE)
y_val_t = torch.from_numpy(y_val).to(DEVICE)

history = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

for epoch in range(1, BEST_CONFIG["epochs"] + 1):
    model.train()
    epoch_loss, n_samples = 0.0, 0
    y_true_ep, y_pred_ep = [], []

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        bs = xb.size(0)
        epoch_loss += loss.item() * bs
        n_samples += bs
        y_true_ep.extend(yb.detach().cpu().numpy().tolist())
        y_pred_ep.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    train_loss = epoch_loss / n_samples
    train_acc = accuracy_score(y_true_ep, y_pred_ep)

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t)
        val_loss = criterion(val_logits, y_val_t).item()
        val_pred = val_logits.argmax(dim=1)
        val_acc = (val_pred == y_val_t).float().mean().item()
        val_f1 = f1_score(y_val_t.cpu().numpy(), val_pred.cpu().numpy(), zero_division=0)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    if epoch % 10 == 0 or epoch == 1 or epoch == BEST_CONFIG["epochs"]:
        logger.info(
            "epoch %d/%d \u2014 train_loss=%.4f train_acc=%.4f val_loss=%.4f val_acc=%.4f val_f1=%.4f",
            epoch, BEST_CONFIG["epochs"], train_loss, train_acc, val_loss, val_acc, val_f1,
        )

history_df = pd.DataFrame(history)
print("\n=== Ringkasan 5 epoch terakhir ===")
print(history_df.tail(5).to_string(index=False))

## Bagian 5 — Learning Curve

Plot loss & F1 (train vs val) per epoch, disimpan ke file PNG (konsisten konvensi project — plot
tidak ditampilkan inline).

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PLOT_DIR = PROJECT_ROOT / "notebooks"

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (CrossEntropy)")
axes[0].set_title("Learning Curve \u2014 Loss")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
axes[1].plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
axes[1].plot(history_df["epoch"], history_df["val_f1"], label="val_f1", linestyle="--")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Skor")
axes[1].set_title("Learning Curve \u2014 Accuracy & F1")
axes[1].legend()

fig.tight_layout()
fig.savefig(PLOT_DIR / "eda08_learning_curve.png", dpi=100)
plt.close(fig)
print(f"Learning curve disimpan: {PLOT_DIR / 'eda08_learning_curve.png'}")

## Bagian 6 — Evaluasi Test-Set (HANYA 4 Metrik)

Evaluasi model hasil training final pada test-set. Sesuai Batasan Masalah BAB I: **HANYA** Accuracy,
Precision, Recall, F1-Score. Confusion matrix mentah ditampilkan sebagai pendukung (bukan metrik
tambahan).

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

X_test_t = torch.from_numpy(X_test).to(DEVICE)

model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    y_test_pred = test_logits.argmax(dim=1).cpu().numpy()

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred, labels=[0, 1]).ravel()

final_metrics = {
    "accuracy": accuracy_score(y_test, y_test_pred),
    "precision": precision_score(y_test, y_test_pred, zero_division=0),
    "recall": recall_score(y_test, y_test_pred, zero_division=0),
    "f1": f1_score(y_test, y_test_pred, zero_division=0),
}
final_confusion = {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

print("=== Model Final (Notebook 08) \u2014 Test Set ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.6f}")
print("Confusion matrix:", final_confusion)

## Bagian 7 — Sanity Check vs Hasil Random Search Notebook 07

Config & seed dasar sama dengan iterasi pemenang notebook 07 — hasil training ulang ini diharapkan
MIRIP (bukan harus identik persis, karena `DataLoader(shuffle=True)` & non-determinism CUDA bisa
menggeser urutan batch antar run) dengan metrik yang tercatat di `finbert_lstm_skripsi_metadata.json`.
Selisih besar mengindikasikan ada perbedaan preprocessing/config yang perlu ditelusuri.

In [ ]:
comparison_df = pd.DataFrame({
    "Metrik": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Notebook 07 (tercatat)": [
        random_search_metadata["metrics"]["accuracy"],
        random_search_metadata["metrics"]["precision"],
        random_search_metadata["metrics"]["recall"],
        random_search_metadata["metrics"]["f1"],
    ],
    "Notebook 08 (training ulang)": [
        final_metrics["accuracy"],
        final_metrics["precision"],
        final_metrics["recall"],
        final_metrics["f1"],
    ],
})
comparison_df["Selisih"] = (
    comparison_df["Notebook 08 (training ulang)"] - comparison_df["Notebook 07 (tercatat)"]
)
print(comparison_df.to_string(index=False))

## Bagian 8 — Simpan Artefak

Nama file terisolasi (`finbert_lstm_notebook08_final.*`) — **TIDAK** menimpa artefak notebook 07
(`finbert_lstm_skripsi.*`) maupun artefak produksi lain. Disimpan di folder yang sama
(`data/artifacts/lstm_weights/`) mengikuti konvensi penamaan project ini.

In [ ]:
import joblib
from datetime import datetime, timezone

ARTIFACT_DIR = PROJECT_ROOT / "data" / "artifacts" / "lstm_weights"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACT_NAME = "finbert_lstm_notebook08_final"
model_path = ARTIFACT_DIR / f"{ARTIFACT_NAME}.pt"
scaler_path = ARTIFACT_DIR / f"{ARTIFACT_NAME}_scaler.joblib"
metadata_path = ARTIFACT_DIR / f"{ARTIFACT_NAME}_metadata.json"

torch.save({
    "model_state_dict": model.state_dict(),
    "config": BEST_CONFIG,
    "feature_cols": FEATURE_COLS,
    "window": WINDOW,
}, model_path)

joblib.dump(scaler, scaler_path)

metadata = {
    "name": ARTIFACT_NAME,
    "source_random_search_metadata": str(RANDOM_SEARCH_METADATA_PATH.name),
    "feature_cols": FEATURE_COLS,
    "window": WINDOW,
    "horizon": HORIZON,
    "config": BEST_CONFIG,
    "train_seed": TRAIN_SEED,
    "metrics": final_metrics,
    "confusion_matrix": final_confusion,
    "history": history,
    "split_sizes": {
        "train_rows": len(df_train),
        "val_rows": len(df_val),
        "test_rows": len(df_test),
        "train_samples": len(X_train),
        "val_samples": len(X_val),
        "test_samples": len(X_test),
    },
    "trained_at": datetime.now(timezone.utc).isoformat(),
}
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Saved: {model_path.name}, {scaler_path.name}, {metadata_path.name}")